# 1. Імпорт бібліотек

In [1]:
import torch
import pandas as pd
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print(f"PyTorch версія: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

e:\conda_envs\xlm-bert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\conda_envs\xlm-bert\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


PyTorch версія: 2.6.0+cu124
CUDA доступна: True


# 2. Конфігурація та завантаження даних

In [2]:

FILE_PATH = "../../data/final_dataset.csv"
MODEL_NAME = "bert-base-multilingual-cased"
TEXT_COLUMN = "text"
LABEL_COLUMN = "fake"
MAX_LENGTH = 256

print(f"Завантаження даних з {FILE_PATH}...")
df = pd.read_csv(FILE_PATH)

df = df.rename(columns={LABEL_COLUMN: 'label'})
df = df[[TEXT_COLUMN, 'label']]

dataset = Dataset.from_pandas(df)

print("Дані завантажено:")
print(dataset)

Завантаження даних з ../../data/final_dataset.csv...
Дані завантажено:
Dataset({
    features: ['text', 'label'],
    num_rows: 9988
})


# 3. Розділення даних (70/15/15)

In [3]:
train_test_split = dataset.train_test_split(test_size=0.3, seed=42)
test_valid_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_valid_split['train'],
    'test': test_valid_split['test']
})

print("Дані розділено:")
print(dataset_dict)

Дані розділено:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6991
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1498
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1499
    })
})


# 4. Завантаження моделі та токенізатора

In [4]:
print(f"Завантаження токенізатора для {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Завантаження моделі {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Модель завантажена на: {device}")

Завантаження токенізатора для bert-base-multilingual-cased...
Завантаження моделі bert-base-multilingual-cased...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Модель завантажена на: cuda


# 5. Токенізація

In [5]:
def tokenize_function(examples):
    return tokenizer(
        examples[TEXT_COLUMN], 
        padding="max_length", 
        truncation=True, 
        max_length=MAX_LENGTH
    )

print("Токенізація датасету...")
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COLUMN])
print("Токенізація завершена.")

Токенізація датасету...


Map:   0%|          | 0/6991 [00:00<?, ? examples/s]

Map: 100%|██████████| 1499/1499 [00:00<00:00, 7576.60 examples/s]

Токенізація завершена.


# 6. Функція для розрахунку метрик

In [6]:
from sklearn.metrics import fbeta_score

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)
    f2 = fbeta_score(labels, predictions, beta=2)
    
    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f2": f2
    }

print("Функція метрик готова.")

Функція метрик готова.


# 7. Налаштування та запуск навчання

In [7]:
training_args = TrainingArguments(
    output_dir="../../data/results_mbert",
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Починаємо тренування...")
trainer.train()
print("Тренування завершено.")

C:\Users\kaval\AppData\Local\Temp\ipykernel_41712\1992908924.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Починаємо тренування...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F2
1,0.078800,0.026168,0.995327,0.994660,0.995989,0.995723
2,0.005900,0.023509,0.994660,0.989418,1.000000,0.997866
3,0.020200,0.055103,0.991322,0.982917,1.000000,0.996536
4,0.000000,0.023225,0.997330,0.994681,1.000000,0.998932
5,0.000000,0.023366,0.995995,0.995989,0.995989,0.995989


Тренування завершено.


# 8. Оцінка на тестовій вибірці

In [8]:
print(" ОЦІНКА НА ТЕСТОВІЙ ВИБІРЦІ ".center(50, "="))

test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("Результати на тестовій вибірці:\n")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"F2-score:  {test_results['eval_f2']:.4f}")

=========== ОЦІНКА НА ТЕСТОВІЙ ВИБІРЦІ ===========


Результати на тестовій вибірці:

Accuracy:  0.9960
Precision: 0.9923
Recall:    1.0000
F2-score:  0.9984
